## Rotated Surface Code: phase-diagram sweep at $d = 3, 4, 5$

Drop-in Colab notebook.  KD sweep of $h_z \in [0, 0.75]$ at fixed $h_x = 0.3$ on the rotated surface code for code distances $d = 3, 4, 5$.

**Runtime.**  Use **Runtime → Change runtime type → CPU, High-RAM**.  The bottleneck is `scipy.sparse.linalg.eigsh`, which runs on CPU; a GPU runtime (T4 / A100) actively *hurts* because JAX/XLA tries to put NetKet's sparse Hamiltonian on GPU memory and OOMs.  The next cell pins JAX to CPU so this works even if you forget.

**Per-system rough costs (real dtype, on CPU):**

| $d$ | qubits | Hilbert dim | sparse $H$ | feasible on |
|---|---|---|---|---|
| 3 | 9  | 512        | KB    | free CPU |
| 4 | 16 | 65 k       | tens of MB | free CPU |
| 5 | 25 | 33 M       | ~30 GB | High-RAM CPU |

On free Colab the $d = 5$ sweep is expected to OOM during sparse-matrix construction; comment it out of the run-cell if that happens.

In [ ]:
!pip install -q netket

In [ ]:
# Pin JAX to CPU *before* importing jax / netket.  Avoids GPU OOM on Colab
# A100 / T4 runtimes — eigsh runs on CPU anyway, GPU buys us nothing here.
import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_PLATFORM_NAME"] = "cpu"

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.sparse.linalg import eigsh
import jax
import netket as nk
from netket.operator.spin import sigmax, sigmaz

print("JAX backend:", jax.default_backend())  # should print 'cpu'

## Geometry

$d \times d$ qubits on integer sites $(i, j)$ linearly indexed as $q(i, j) = i \cdot d + j$.  Bulk: weight-4 $X$- and $Z$-plaquettes on a checkerboard.  Boundary: weight-2 $X$-stabilizers on top/bottom edges, weight-2 $Z$-stabilizers on left/right, included only where the natural checkerboard extends past the lattice with matching colour.

In [ ]:
class RotatedSurfaceGeometry:
    """Rotated surface code on a d x d grid of vertex qubits."""

    def __init__(self, d):
        if d < 2:
            raise ValueError("d must be >= 2")
        self.d = d
        self.N = d * d
        self.x_stabs, self.z_stabs = self._build_stabilizers()

    def _q(self, i, j):
        return i * self.d + j

    def _build_stabilizers(self):
        d = self.d
        x_stabs, z_stabs = [], []

        # Bulk weight-4 plaquettes in a checkerboard.
        for i in range(d - 1):
            for j in range(d - 1):
                face = [self._q(i, j),     self._q(i + 1, j),
                        self._q(i, j + 1), self._q(i + 1, j + 1)]
                (x_stabs if (i + j) % 2 == 0 else z_stabs).append(face)

        # Boundary weight-2 stabilizers: virtual face just outside the lattice
        # contributes a stabilizer of its checkerboard colour.
        for i in range(d - 1):
            if (i + (d - 1)) % 2 == 0:
                x_stabs.append([self._q(i, d - 1), self._q(i + 1, d - 1)])
            if (i - 1) % 2 == 0:
                x_stabs.append([self._q(i, 0), self._q(i + 1, 0)])

        for j in range(d - 1):
            if (j - 1) % 2 == 1:
                z_stabs.append([self._q(0, j), self._q(0, j + 1)])
            if ((d - 1) + j) % 2 == 1:
                z_stabs.append([self._q(d - 1, j), self._q(d - 1, j + 1)])

        return x_stabs, z_stabs


# Sanity print.
for d in (3, 4, 5):
    g = RotatedSurfaceGeometry(d)
    print(f"d={d}: N={g.N}, X-stabs={len(g.x_stabs)}, Z-stabs={len(g.z_stabs)}, E0(expected) = {-len(g.x_stabs) - len(g.z_stabs)}")

## Hamiltonian and observables

$$H = -J \sum_v A_v - J \sum_p B_p - h_x \sum_i \sigma^x_i - h_z \sum_i \sigma^z_i.$$

No $\sigma^y$ terms, so we keep everything in `float64`.  We also expose the three observables we want to track:  $A = \sum_v A_v$, $B = \sum_p B_p$, $M_z = \sum_i \sigma^z_i$.

In [ ]:
DTYPE = float


def prod_op(group, single):
    op = 1
    for i in group:
        op *= single(i)
    return op


def build_hamiltonian(hi, geom, hx, hz, J=1.0):
    sx = lambda i: sigmax(hi, i, dtype=DTYPE)
    sz = lambda i: sigmaz(hi, i, dtype=DTYPE)
    H = 0
    for star in geom.x_stabs:
        H += -J * prod_op(star, sx)
    for plaq in geom.z_stabs:
        H += -J * prod_op(plaq, sz)
    if hx != 0:
        H += -hx * sum(sx(i) for i in range(geom.N))
    if hz != 0:
        H += -hz * sum(sz(i) for i in range(geom.N))
    return H.to_pauli_strings()


def build_observables(hi, geom):
    sx = lambda i: sigmax(hi, i, dtype=DTYPE)
    sz = lambda i: sigmaz(hi, i, dtype=DTYPE)
    A = sum(prod_op(s, sx) for s in geom.x_stabs)
    B = sum(prod_op(p, sz) for p in geom.z_stabs)
    Mz = sum(sz(i) for i in range(geom.N))
    return A.to_sparse(), B.to_sparse(), Mz.to_sparse()


def expect(O, psi):
    return float((psi.conj() @ O @ psi).real)

## Sweep

Fixed $h_x = 0.3$, sweep $h_z \in [0, 0.75]$ over 20 points.  Krylov diagonalization with `k = 4` so we can track the gap.  Returns a `dict` of arrays.

In [ ]:
def sweep_phase_diagram(geom, h_z_list, hx=0.3, k=4, label=""):
    hi = nk.hilbert.Spin(s=1 / 2, N=geom.N)
    A_sp, B_sp, Mz_sp = build_observables(hi, geom)
    N_v, N_p, N = len(geom.x_stabs), len(geom.z_stabs), geom.N

    records = {key: [] for key in ("h_z", "E0", "gap", "A", "B", "Mz")}
    for h_z in tqdm(h_z_list, desc=label):
        H_sp = build_hamiltonian(hi, geom, hx, h_z).to_sparse()
        eig_vals, eig_vecs = eigsh(H_sp, k=k, which="SA", return_eigenvectors=True)
        order = np.argsort(eig_vals)
        eig_vals, eig_vecs = eig_vals[order], eig_vecs[:, order]
        psi = eig_vecs[:, 0]

        records["h_z"].append(h_z)
        records["E0"].append(eig_vals[0])
        records["gap"].append(eig_vals[1] - eig_vals[0])
        records["A"].append(expect(A_sp, psi) / N_v)
        records["B"].append(expect(B_sp, psi) / N_p)
        records["Mz"].append(expect(Mz_sp, psi) / N)

    return {key: np.array(v) for key, v in records.items()}

## Run

Comment out `d = 5` if you're on free Colab and the sparse-matrix build OOMs.

In [ ]:
h_z_list = np.linspace(0, 0.75, 20)

runs = {}
for d in (3, 4, 5):
    geom = RotatedSurfaceGeometry(d)
    runs[f"d={d}"] = sweep_phase_diagram(geom, h_z_list, hx=0.3, label=f"d={d}")

## Plot

Four panels: $\langle A_v \rangle$, $\langle B_p \rangle$, gap $E_1 - E_0$, and per-site $\langle \sigma^z \rangle$, overlaid for each $d$.  Expect the gap minimum and the $\langle A_v \rangle$ drop to sharpen as $d$ grows.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
panels = [
    ("A",   r"$\langle A_v \rangle$"),
    ("B",   r"$\langle B_p \rangle$"),
    ("gap", r"$E_1 - E_0$"),
    ("Mz",  r"$\langle \sigma^z \rangle$"),
]

for ax, (key, ylabel) in zip(axes.flat, panels):
    for label, rec in runs.items():
        ax.plot(rec["h_z"], rec[key], "o-", markersize=4, label=label)
    ax.set_ylabel(ylabel)
    ax.legend()

for ax in axes[-1]:
    ax.set_xlabel(r"$h_z$")

fig.tight_layout()
plt.show()

## (Optional) Save results

Colab sessions are ephemeral.  If you want to keep the data, mount Drive and dump `runs` to a `.npz` file.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# flat = {f"{lab}_{key}": arr for lab, rec in runs.items() for key, arr in rec.items()}
# np.savez('/content/drive/MyDrive/tc_surface_sweep.npz', **flat)